In [2]:
import pandas as pd
import os

In [ ]:
# Path to the folder containing the CSV files
csv_folder = "Economic-Growth-Prediction"

# Output file
output_file = "viirs_20.csv"

# List of CSV files to merge (all files in the folder)
csv_files = [
    "viirs_20_rio_agg.csv",
    "viirs_20_sao_agg.csv",
    "viirs_20_bahia_agg.csv"
]

# Create an empty list to hold DataFrames
dataframes = []

# Loop through the files and read them
for file in csv_files:
    file_path = os.path.join(csv_folder, file)
    df = pd.read_csv(file_path)

    # Extract state, year, and month from the filename
    # parts = file.split("_")
    # state = parts[1]  # The second part is the state
    # month = parts[2]  # The third part is the month

    # # Add columns for state, year, and month
    # df["state"] = state
    # df["year"] = 2020
    # df["month"] = month

    dataframes.append(df)

# Concatenate all DataFrames
merged_data = pd.concat(dataframes, ignore_index=True)

# Save the merged dataset to a CSV file
merged_data.to_csv(output_file, index=False)

print(f"Merged dataset saved to {output_file}")

Merged dataset saved to D:\NU\semester_5\data_analysis\project\Economic-Growth-Prediction\viirs_year_dataset.csv


In [ ]:
output_file = "viirs_20.csv"

df = pd.read_csv(output_file)
df

           longitude   latitude  light_intensity  state  year  month
0         -44.925000 -20.837500              0.0    rio  2020      1
1         -44.920833 -20.837500              0.0    rio  2020      1
2         -44.916667 -20.837500              0.0    rio  2020      1
3         -44.912500 -20.837500              0.0    rio  2020      1
4         -44.908333 -20.837500              0.0    rio  2020      1
...              ...        ...              ...    ...   ...    ...
103566079 -37.370833 -18.316667              0.0  bahia  2020     12
103566080 -37.366666 -18.316667              0.0  bahia  2020     12
103566081 -37.362500 -18.316667              0.0  bahia  2020     12
103566082 -37.358333 -18.316667              0.0  bahia  2020     12
103566083 -37.354166 -18.316667              0.0  bahia  2020     12

[103566084 rows x 6 columns]


In [4]:
df.duplicated().sum()

0

In [5]:
# Step 1: Drop the 'state' column and check for duplicates
duplicated_mask = df.drop(columns=['state']).duplicated()

# Step 2: Get the duplicated rows
duplicated_rows = df[duplicated_mask]

# Step 3: Output the result
print("Duplicated Rows (Ignoring 'state'):")
print(duplicated_rows)

# Step 4: Count of duplicates
print(f"Number of duplicates: {duplicated_rows.shape[0]}")

Duplicated Rows (Ignoring 'state'):
          longitude   latitude  light_intensity state  year  month
7680740  -44.920833 -20.837500              0.0   sao  2020      1
7680741  -44.916667 -20.837500              0.0   sao  2020      1
7680742  -44.912500 -20.837500              0.0   sao  2020      1
7680743  -44.908333 -20.837500              0.0   sao  2020      1
7680744  -44.904167 -20.837500              0.0   sao  2020      1
...             ...        ...              ...   ...   ...    ...
40226839 -44.258333 -23.441667              0.0   sao  2020     12
40226840 -44.254167 -23.441667              0.0   sao  2020     12
40226841 -44.250000 -23.441667              0.0   sao  2020     12
40226842 -44.245833 -23.441667              0.0   sao  2020     12
40226843 -44.241667 -23.441667              0.0   sao  2020     12

[650061 rows x 6 columns]
Number of duplicates: 650061


In [ ]:
# Group and inspect duplicates for further analysis
duplicate_summary = duplicated_rows.groupby(['longitude', 'latitude', 'year', 'month']).size()
duplicate_summary

longitude   latitude    year  month
-44.920833  -23.441667  2020  11       1
                              12       1
            -23.437500  2020  11       1
                              12       1
            -23.433334  2020  11       1
                                      ..
-44.241667  -20.837500  2020  8        1
                              9        1
                              10       1
                              11       1
                              12       1
Length: 650061, dtype: int64


In [ ]:
# Filter locations with more than one duplicate
locations_with_duplicates = duplicate_summary[duplicate_summary > 1]
locations_with_duplicates

Series([], dtype: int64)


In [ ]:
# Check if 'state' differentiates duplicates
unique_states_per_group = duplicated_rows.groupby(['longitude', 'latitude', 'year', 'month'])['state'].nunique()
unique_states_per_group[unique_states_per_group > 1]

Series([], Name: state, dtype: int64)


In [ ]:
# Check duplicates within each state
duplicates_per_state = df.groupby('state').apply(lambda x: x.duplicated(subset=['longitude', 'latitude', 'year', 'month']).sum())
print("Number of duplicates per state:")
duplicates_per_state

Number of duplicates per state:
state
bahia    0
rio      0
sao      0
dtype: int64


C:\Users\PC\AppData\Local\Temp\ipykernel_17608\2699302351.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicates_per_state = df.groupby('state').apply(lambda x: x.duplicated(subset=['longitude', 'latitude', 'year', 'month']).sum())


In [10]:
# Find rows with cross-state duplicates
cross_state_duplicates = df[df.duplicated(subset=['longitude', 'latitude', 'year', 'month'], keep=False)]
print("Cross-state duplicates:")
print(cross_state_duplicates)

Cross-state duplicates:
          longitude   latitude  light_intensity state  year  month
1        -44.920833 -20.837500              0.0   rio  2020      1
2        -44.916667 -20.837500              0.0   rio  2020      1
3        -44.912500 -20.837500              0.0   rio  2020      1
4        -44.908333 -20.837500              0.0   rio  2020      1
5        -44.904167 -20.837500              0.0   rio  2020      1
...             ...        ...              ...   ...   ...    ...
40226839 -44.258333 -23.441667              0.0   sao  2020     12
40226840 -44.254167 -23.441667              0.0   sao  2020     12
40226841 -44.250000 -23.441667              0.0   sao  2020     12
40226842 -44.245833 -23.441667              0.0   sao  2020     12
40226843 -44.241667 -23.441667              0.0   sao  2020     12

[1908048 rows x 6 columns]


In [11]:
# Inspect distinct states for each duplicate group
unique_states_per_group = cross_state_duplicates.groupby(['longitude', 'latitude', 'year', 'month'])['state'].nunique()
print(unique_states_per_group[unique_states_per_group > 1])

longitude   latitude    year  month
-44.920833  -23.441667  2020  1        2
                              2        2
                              3        2
                              4        2
                              5        2
                                      ..
-44.241667  -20.837500  2020  8        2
                              9        2
                              10       2
                              11       2
                              12       2
Name: state, Length: 954024, dtype: int64


In [ ]:
# Combine states for each duplicate group
aggregated_data = cross_state_duplicates.groupby(['longitude', 'latitude', 'year', 'month'], as_index=False).agg({
    'light_intensity': 'sum',  # Optionally sum light intensity
    'state': ', '.join  # Combine state values into a single string
})
print(f"Aggregated dataset shape: {aggregated_data.shape}")

Aggregated dataset shape: (954024, 6)


In [ ]:
aggregated_data.head()

   longitude   latitude  year  month  light_intensity     state
0 -44.920833 -23.441667  2020      1             0.06  rio, sao
1 -44.920833 -23.441667  2020      2             0.29  rio, sao
2 -44.920833 -23.441667  2020      3             0.23  rio, sao
3 -44.920833 -23.441667  2020      4             0.33  rio, sao
4 -44.920833 -23.441667  2020      5             0.29  rio, sao
